# GPT 5.4 Full Run

In [0]:
import pandas as pd
manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)

checklist_filtered = pd.read_csv("/Workspace/Users/user/user/checklist_filtered.csv")
display(checklist_filtered)

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("my-agent-secrets")

In [0]:
import openai
import pandas as pd
from openai import AzureOpenAI

import os
import pandas as pd
import numpy as np
import time
import json
import ast
import copy
from IPython.display import IFrame
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import re

In [0]:
api_key = dbutils.secrets.get(scope="my-agent-secrets", key="openai-api-key")
#api_version = "2026-03-05"
api_version = "2025-01-01-preview"
azure_endpoint = dbutils.secrets.get(scope="my-agent-secrets", key="openai-endpoint")
model = "gpt-5.4"

In [0]:
from openai import OpenAI

client = AzureOpenAI(
    api_key = api_key,
    api_version = api_version,
    azure_endpoint = azure_endpoint
)

In [0]:
# Test 1: Check your client config
print(f"API Key set: {bool(client.api_key)}")
print(f"API Key (last 4): ...{client.api_key[-4:]}")
print(f"Base URL: {client.base_url}")
print(f"API Version: {client._api_version}")



In [0]:
# Test 2: Send a simple test message
try:
    response = client.chat.completions.create(
        model= model,
        messages=[{"role": "user", "content": "Say hello"}]
    )
    print(f"\nConnection successful!")
    print(f"Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"\nConnection failed:")
    print(f"Error type: {type(e).__name__}")
    print(f"Error: {e}")

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

results_GPT_full = []

total_prompt_tokens = 0
total_completion_tokens = 0

for idx, row in notes_enrolled4.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    total_prompt_tokens += response.usage.prompt_tokens
    total_completion_tokens += response.usage.completion_tokens
    
    try:
        output = json.loads(response.choices[0].message.content)
        phenos = output.get("Phenotypes", {})
        for pheno_name, details in phenos.items():
            results_GPT_full.append({
                "Subject ID": subject_id,
                "Phenotype": pheno_name,
                "Score": details.get("score", 0),
                "Justification": details.get("justification", ""),
                "Quote": details.get("quote", "")
            })
    except json.JSONDecodeError:
        print(f"Failed to parse response for {subject_id} (row {idx})")
        results_GPT_full.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")

print(f"Prompt tokens: {total_prompt_tokens:,}")
print(f"Completion tokens: {total_completion_tokens:,}")
print(f"Total tokens: {total_prompt_tokens + total_completion_tokens:,}")

# Save results_GPT_full — now in long format (one row per subject × phenotype)
results_GPT_full_df = pd.DataFrame(results_GPT_full)
results_GPT_full_df.to_json(
    "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full.json",
    orient="records", indent=2
)
print(f"Saved {len(results_GPT_full_df)} rows")
display(results_GPT_full_df.head(10))

Processed 10 notes...

Processed 20 notes...

Processed 30 notes...

Processed 40 notes...

Processed 50 notes...

Processed 60 notes...

Processed 70 notes...

Processed 80 notes...

Processed 90 notes...

Processed 100 notes...

Processed 110 notes...

RateLimitError: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
Error(message=Cannot find child abc, errno=no such file or directory, statusCode=0)
[Trace ID: _________]

In [0]:
raw = client.chat.completions.with_raw_response.create(
    model=model,
    messages=[{"role": "user", "content": "Hello"}],
)

print("status:", raw.status_code)
for k, v in raw.headers.items():
    if "ratelimit" in k.lower() or k.lower() == "retry-after":
        print(k, "=", v)

resp = raw.parse()   # the normal response object, if you want it

In [0]:
(notes_enrolled4['NOTE_TEXT'].str.len().max() // 4)   # rough max note tokens; add ~system-prompt + reserve

In [0]:
TPM_LIMIT = 10_000          # from x-ratelimit-limit-tokens
COMPLETION_RESERVE = 1500   # rough tokens reserved for the JSON output

sys_tokens = len(system_prompt) // 4
note_budget_chars = (TPM_LIMIT - sys_tokens - COMPLETION_RESERVE) * 4
note_lens = notes_enrolled4['NOTE_TEXT'].str.len()
over = note_lens > note_budget_chars

print(f"system prompt ≈ {sys_tokens} tokens")
print(f"safe note budget ≈ {note_budget_chars // 4} tokens ({note_budget_chars} chars)")
print(f"{over.sum()} of {len(note_lens)} notes exceed the window and will fail as-is")

In [0]:
import tiktoken
try:
    enc = tiktoken.encoding_for_model("gpt-5.4")
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")   # recent-GPT encoding; close proxy

count = lambda t: len(enc.encode(t))

sys_tokens  = count(system_prompt)
note_tokens = notes_enrolled4['NOTE_TEXT'].apply(count)

HARD_LIMIT, COMPLETION_RESERVE, SPLIT_MARGIN = 10_000, 1500, 0.95
budget_tokens = int((HARD_LIMIT - sys_tokens - COMPLETION_RESERVE) * SPLIT_MARGIN)
flagged_mask  = note_tokens > budget_tokens
print(f"{flagged_mask.sum()} notes to set aside (budget ≈ {budget_tokens} note tokens)")

In [0]:
oversized_df = notes_enrolled4.loc[flagged_mask].copy()    # set aside / run after limit raised
fits_df      = notes_enrolled4.loc[~flagged_mask].copy()   # safe to run now

print(f"oversized_df: {len(oversized_df)} notes")
print(f"fits_df:      {len(fits_df)} notes")

In [0]:
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full1.json"
SAVE_EVERY = 10
saved_count = 0

results_GPT_full1 = []

total_prompt_tokens = 0
total_completion_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_GPT_full1).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in fits_df.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    total_prompt_tokens += response.usage.prompt_tokens
    total_completion_tokens += response.usage.completion_tokens
    
    try:
        output = json.loads(response.choices[0].message.content)
        phenos = output.get("Phenotypes", {})
        for pheno_name, details in phenos.items():
            results_GPT_full1.append({
                "Subject ID": subject_id,
                "Phenotype": pheno_name,
                "Score": details.get("score", 0),
                "Justification": details.get("justification", ""),
                "Quote": details.get("quote", "")
            })
    except json.JSONDecodeError:
        print(f"Failed to parse response for {subject_id} (row {idx})")
        results_GPT_full1.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")
        
    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Saved progress — {saved_count} notes done")
        
save_progress()
print(f"Prompt tokens: {total_prompt_tokens:,}")
print(f"Completion tokens: {total_completion_tokens:,}")
print(f"Total tokens: {total_prompt_tokens + total_completion_tokens:,}")

# Save results_GPT_full1 — now in long format (one row per subject × phenotype)
results_GPT_full_df1 = pd.DataFrame(results_GPT_full1)
results_GPT_full_df1.to_json(
    "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full1.json",
    orient="records", indent=2
)
print(f"Saved {len(results_GPT_full_df1)} rows")
display(results_GPT_full_df1.head(10))

Prompt tokens: 3,803,663

Completion tokens: 2,834,495

Total tokens: 6,638,158

Saved 81040 rows

Took around 7 hours

In [0]:
# Running the larger notes now that our tokens per minute limit has been increased
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full2.json"
SAVE_EVERY = 10
saved_count = 0

results_GPT_full2 = []

total_prompt_tokens = 0
total_completion_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_GPT_full2).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in oversized_df.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    total_prompt_tokens += response.usage.prompt_tokens
    total_completion_tokens += response.usage.completion_tokens
    
    try:
        output = json.loads(response.choices[0].message.content)
        phenos = output.get("Phenotypes", {})
        for pheno_name, details in phenos.items():
            results_GPT_full2.append({
                "Subject ID": subject_id,
                "Phenotype": pheno_name,
                "Score": details.get("score", 0),
                "Justification": details.get("justification", ""),
                "Quote": details.get("quote", "")
            })
    except json.JSONDecodeError:
        print(f"Failed to parse response for {subject_id} (row {idx})")
        results_GPT_full2.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")
        
    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Saved progress — {saved_count} notes done")
        
save_progress()
print(f"Prompt tokens: {total_prompt_tokens:,}")
print(f"Completion tokens: {total_completion_tokens:,}")
print(f"Total tokens: {total_prompt_tokens + total_completion_tokens:,}")

# Save results_GPT_full2 — now in long format (one row per subject × phenotype)
results_GPT_full_df2 = pd.DataFrame(results_GPT_full2)
results_GPT_full_df2.to_json(
    "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full2.json",
    orient="records", indent=2
)
print(f"Saved {len(results_GPT_full_df2)} rows")
display(results_GPT_full_df2.head(10))

Prompt tokens: 148,751

Completion tokens: 76,929

Total tokens: 225,680

Saved 2000 rows

In [0]:
import pandas as pd

results_GPT_full_df1 = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full1.json", orient="records")
results_GPT_full_df1['Phenotype'] = results_GPT_full_df1['Phenotype'].str.strip()
display(results_GPT_full_df1)
print(len(results_GPT_full_df1))

results_GPT_full_df2 = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full2.json", orient="records")
results_GPT_full_df2['Phenotype'] = results_GPT_full_df2['Phenotype'].str.strip()
display(results_GPT_full_df2)
print(len(results_GPT_full_df2))

manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)
print(len(notes_enrolled4))

In [0]:
results_GPT_full_df = pd.concat([results_GPT_full_df1, results_GPT_full_df2], ignore_index=True)
results_GPT_full_df.to_csv("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full.csv", index=False)
print(f"Combined shape: {results_GPT_full_df.shape}")
display(results_GPT_full_df)
print(len(results_GPT_full_df))
print(f"{len(results_GPT_full_df1)} + {len(results_GPT_full_df2)} = {len(results_GPT_full_df1) + len(results_GPT_full_df2)}")

In [0]:
subjects_to_drop = [
    6 WES IDs
]
display(notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_to_drop)])

In [0]:
subjects_incomplete = (
    results_GPT_full_df.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
display(subjects_incomplete)

for subject_id in subjects_incomplete['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_GPT_full_df[results_GPT_full_df['Subject ID'] == subject_id])

In [0]:
print(subjects_incomplete)

In [0]:
rerun = notes_enrolled4[notes_enrolled4['Subject ID'].isin(subjects_incomplete['Subject ID'])]
display(rerun)

In [0]:
# Running the larger notes now that our tokens per minute limit has been increased
# phenotype-specific justification
import json

phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)


phenotypes = checklist_filtered['Term from checklist'].dropna().unique().tolist()

phenotype_list_str = "\n".join(f"- {p}" for p in phenotypes)

system_prompt = f"""You are a world class board-certified medical geneticist performing a structured chart review.

TASK: Read the provided clinical note and determine whether each phenotype below is present (1) or absent (0) for the patient.

RULES:
- Output ONLY a valid JSON object and nothing else.
- The JSON must contain exactly one key: "Phenotypes".
- "Phenotypes": An object where each key is a phenotype name exactly as listed below, and each value is an object with three keys:
  - "score": 0 or 1
  - "justification": Why THIS SPECIFIC phenotype is present or absent. Do not summarize the entire note. Explain what evidence in the note directly relates to this phenotype, or state that no relevant evidence was found.
  - "quote": The exact word-for-word excerpt from the note that supports your decision for THIS SPECIFIC phenotype. Copy the text exactly as written in the note. Use "" if no relevant text exists.
- Each phenotype's justification and quote must pertain ONLY to that phenotype. Do not reuse the same justification or quote across multiple phenotypes unless the exact same sentence genuinely supports both.

SCORING CRITERIA:
- Use 1 ONLY if the note provides clear evidence that the patient HAS the phenotype as a clinically significant finding — a confirmed diagnosis, an objective measurement that meets a clinical threshold for pathology, or symptoms explicitly identified as pathological by the clinician.
- Use 0 if the note does not mention the phenotype, explicitly rules it out, there is insufficient evidence, or a reported value does not meet the accepted clinical threshold for that phenotype.
- If the note mentions a phenotype was historically present but has since resolved, still code as 1.
- Score based ONLY on findings documented for THIS patient. If the note describes features generally associated with a genotype, syndrome, or condition but does not confirm that this specific patient exhibits them, do not score those features as present.

CONTINUOUS AND QUANTITATIVE FEATURES:
- For any phenotype involving a measurable value, do not score based on being merely non-average.
- Score 1 ONLY if the value meets or exceeds the accepted clinical threshold for that phenotype based on CDC and WHO guidelines. For growth parameters, this generally means at or beyond the 3rd/97th percentile (±2 SD) for age and sex.
- If the note provides a raw value with the patient's age and sex, assess whether it crosses the clinical threshold. If the note already states a percentile, use that directly.
- If a value crosses the clinical threshold only marginally — sitting just past the cutoff or hovering at the borderline — do NOT score it 1 on the strength of a single reading. In that borderline case, score 1 only if the note documents a history of the finding: it has been observed on more than one occasion, or is described as persistent, recurrent, chronic, or longstanding. A lone borderline reading with no such documented history should be scored 0. Values that fall clearly and substantially beyond the threshold remain 1 regardless of whether a prior history is noted.
- A value that is below or above average but still within the normal range does NOT qualify as a phenotype. A developmental milestone that is slightly late but within the accepted normal window does NOT qualify as delayed.
- State the threshold you applied in your justification so your reasoning is transparent.

DETERMINING AGE AT A DEVELOPMENTAL MILESTONE:
- Before scoring any developmental delay, establish the patient's age at the milestone, then compare it to that milestone's threshold age.
- If the note records when the milestone occurred, use that. When it gives a date rather than an age, derive the age at the milestone from the age at the visit and the interval between the milestone date and the visit date: age at milestone = age at visit minus the time from the milestone date to the visit date. For example, if the patient is 24 months old at the visit and the milestone occurred 6 months before the visit, the age at the milestone is 18 months.
- If no date or age for the milestone is recorded, assume the age at the milestone is the patient's age at the visit. Apply this visit-age assumption to a milestone the child has not yet met; if a milestone has clearly been achieved and only its timing is unknown, score the delay 0, since you cannot establish it was reached late.
- State in your justification which age you used and whether it came from a recorded milestone date or from the visit age.

AVOIDING OVER-SCORING:
- Do NOT score normal human variation as a phenotype. Mild or subtle physical differences are normal variants unless the clinician explicitly flags them as dysmorphic or clinically significant.
- Do NOT score common benign findings as phenotypes unless the clinician identifies the finding as clinically significant, extensive, or relevant to a diagnosis.
- Qualifiers like "mild," "slight," "subtle," or "minor" should raise your threshold — these findings should only be scored as present if the clinician explicitly connects them to a diagnosis, syndrome, or clinical concern.
- Conversely, if the clinician carries a physical observation into their diagnostic reasoning — naming it in the assessment, impression, clinical concern, or differential — score it 1, even when it was first noted only in passing during the exam. The clinician tying a finding to their diagnostic thinking is itself the signal that it is clinically significant.
- When in doubt, score 0. It is better to miss a borderline finding than to falsely attribute a phenotype based on normal variation.

FACIAL DYSMORPHISM AND EAR MALFORMATION:
- Score facial dysmorphism or ear malformation only when the finding is clinically relevant. Many facial and ear characteristics reflect normal individual variation and are not, on their own, diagnostically meaningful.
- A dysmorphism or malformation that the clinician states plainly as clinically significant, or names as a specific recognized anomaly (e.g., microtia, cleft lip or palate, coloboma), is scored 1 regardless of where in the note it appears. 
- For borderline or subtle facial or ear findings, score 1 ONLY if the clinician acknowledges the dysmorphism or malformation as part of their diagnostic reasoning — in the assessment, impression, clinical concern, or differential — rather than recording it merely as an isolated observation in the physical exam. A borderline feature noted only on physical exam, without being connected to the clinician's diagnostic thinking, should be scored 0.

NON-EXHAUSTIVE EXAMPLES OF THESE PRINCIPLES:
The following illustrate how the above rules apply in practice. These are examples only — apply the same reasoning to ALL phenotypes, not just those mentioned here.
- A "slightly recessed jaw" or "mild facial asymmetry" noted on exam does not constitute facial dysmorphism unless the clinician identifies it as dysmorphic or links it to a syndrome.
- A birthmark or single café-au-lait spot mentioned in passing does not constitute a skin pigmentation phenotype unless the clinician describes it as clinically significant, numerous, or relevant to a diagnosis.
- Height at the 15th percentile is below average but within normal range and does not qualify as short stature. Height at or below the 3rd percentile does.
- A child speaking first words at 14 months is within the normal developmental window and does not qualify as speech delay, even if slightly later than the population mean.
- Low-set or mildly prominent ears noted only under the physical exam, with no further mention, are a borderline finding and should be scored 0. If the clinician carries that finding into their diagnostic reasoning — e.g., listing "low-set, posteriorly rotated ears" among the dysmorphic features supporting a suspected syndrome in the assessment — score 1. Marked microtia or a clearly malformed pinna is scored 1 regardless of where it is documented.
- A QTc interval must be read against age-appropriate norms before it is scored as a long-QT or arrhythmia phenotype: neonates and infants normally have a longer QTc than older children, and most borderline neonatal prolongation resolves on follow-up. A single borderline QTc — roughly 450-470 ms in an infant, or 431-450 ms in an older child — should be scored 0 unless the note documents that the prolongation persisted on a repeat ECG or that long QT was diagnosed. A clearly prolonged QTc (>=500 ms at any age) or an explicit long-QT/LQTS diagnosis is scored 1 regardless of whether a history is given.
- Speech/language delay is judged against CDC milestone ages (the age by which ~75% of children show the skill): no first words besides "mama"/"dada" by 18 months, fewer than three words by 18 months, no two-word combinations (e.g., "more milk") by 24 months, or fewer than ~50 words by 30 months each meets the threshold for delay. A child saying first words at 16 months, or combining two words by 22 months, is within the normal window and does NOT qualify.
- Gross motor delay is judged against CDC/WHO milestone ages: not sitting without support by 9 months, or not walking independently ("without holding on to anyone or anything") by 18 months — the upper bound of the WHO walking window — meets the threshold. A child walking at 13-15 months is within the normal window and does NOT qualify. Absence of crawling is NOT a delay, as crawling is not a CDC milestone.
- Fine motor delay is judged against CDC milestone ages: no pincer grasp (picking up a small object between thumb and forefinger) by 12 months, or inability to hold an object in one hand while manipulating it with the other by 24 months, meets the threshold. An immature raking grasp at 9 months is age-appropriate and does NOT qualify.

PHENOTYPES:
{phenotype_list_str}

EXAMPLE OUTPUT FORMAT:
{{"Phenotypes": {{"Prematurity": {{"score": 0, "justification": "The note does not mention gestational age or premature birth.", "quote": ""}}, "Seizures": {{"score": 1, "justification": "The note documents a history of generalized tonic-clonic seizures beginning at age 3.", "quote": "Patient experienced generalized tonic-clonic seizures starting at age 3, currently managed with levetiracetam."}}, ...}}}}"""

OUTPUT_PATH = "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_rerun.json"
SAVE_EVERY = 10
saved_count = 0

results_GPT_full_rerun = []

total_prompt_tokens = 0
total_completion_tokens = 0
saved_count = 0

def save_progress():
    pd.DataFrame(results_GPT_full_rerun).to_json(OUTPUT_PATH, orient="records", indent=2)

for idx, row in rerun.iterrows():
    subject_id = row['Subject ID']
    note_text = row['NOTE_TEXT']
    
    user_message = f"Clinical note for patient {subject_id}:\n\n{note_text}"
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )
    
    total_prompt_tokens += response.usage.prompt_tokens
    total_completion_tokens += response.usage.completion_tokens
    
    try:
        output = json.loads(response.choices[0].message.content)
        phenos = output.get("Phenotypes", {})
        for pheno_name, details in phenos.items():
            results_GPT_full_rerun.append({
                "Subject ID": subject_id,
                "Phenotype": pheno_name,
                "Score": details.get("score", 0),
                "Justification": details.get("justification", ""),
                "Quote": details.get("quote", "")
            })
    except json.JSONDecodeError:
        print(f"Failed to parse response for {subject_id} (row {idx})")
        results_GPT_full_rerun.append({"Subject ID": subject_id, "Phenotype": "PARSE ERROR"})
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1} notes...")
        
    saved_count += 1
    if saved_count % SAVE_EVERY == 0:
        save_progress()
        print(f"Saved progress — {saved_count} notes done")
        
save_progress()
print(f"Prompt tokens: {total_prompt_tokens:,}")
print(f"Completion tokens: {total_completion_tokens:,}")
print(f"Total tokens: {total_prompt_tokens + total_completion_tokens:,}")

# Save results_GPT_full_rerun — now in long format (one row per subject × phenotype)
results_GPT_full_df_rerun = pd.DataFrame(results_GPT_full_rerun)
results_GPT_full_df_rerun.to_json(
    "/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_rerun.json",
    orient="records", indent=2
)
print(f"Saved {len(results_GPT_full_df_rerun)} rows")
display(results_GPT_full_df_rerun.head(10))

Took 3 minutes

Prompt tokens: 28,073

Completion tokens: 22,027

Total tokens: 50,100

Saved 625 rows

In [0]:
results_GPT_full_df_rerun = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_rerun.json", orient="records")
results_GPT_full_df_rerun['Phenotype'] = results_GPT_full_df_rerun['Phenotype'].str.strip()
display(results_GPT_full_df_rerun)
print(len(results_GPT_full_df_rerun))

In [0]:
import pandas as pd

results_GPT_full_rerun = pd.DataFrame(results_GPT_full_rerun)
subjects_incomplete = (

    results_GPT_full_rerun.groupby('Subject ID')['Phenotype']
    .count()
    .loc[lambda x: x < 125]
    .reset_index()
    .rename(columns={'Phenotype': 'Phenotype Count'})
)

print(f"Subjects with fewer than 125 phenotypes scored: {len(subjects_incomplete)}")
if not subjects_incomplete.empty:    display(subjects_incomplete)


for subject_id in subjects_incomplete['Subject ID']:
    print(f"\nSubject ID: {subject_id}")
    display(results_GPT_full_rerun[results_GPT_full_rerun['Subject ID'] == subject_id])

In [0]:
results_GPT_full_df = pd.concat([
    results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(results_GPT_full_df_rerun['Subject ID'])],
    results_GPT_full_df_rerun
], ignore_index=True)
display(results_GPT_full_df)
print(len(results_GPT_full_df))

In [0]:
print(f"{len(results_GPT_full_df) / 125}")

In [0]:
results_GPT_full_df.to_json("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_final.json", orient="records", indent=2)